In [ ]:
import cogent3
import pandas as pd
import numpy as np
import os
from estimating_ud.config import data_root

In [ ]:
# the coordinate files are read by the eti download wrappers, so they live in the data root, not in results/
folder_in = str(data_root()) + "/"
folder_out = str(data_root() / "intergenic_coordinates") + "/"
os.makedirs(folder_out, exist_ok=True)

table = cogent3.load_table(folder_in + "homo_sapiens-114-gene_metadata.tsv")

In [ ]:
ref_species = "homo_sapiens"
strand = 1
proximal_distance = 5000

chromosomes = list(range(1, 23)) + ['X', 'Y']

for chrom_coord in chromosomes:

    #Calculating whole table for a chromosome
    coordinates_table = table.filtered(lambda x: x == chrom_coord, columns="seqid")
    
    coordinates_table = coordinates_table[:, ["species", "seqid", "start", "stop", "strand"]]
    coordinates_table = coordinates_table.sorted(columns=["start", "stop"])
    coordinates_table.write(folder_out + "chrom" + str(chrom_coord) + "-intragenic.tsv")
    
    start_stop_overlapping = coordinates_table.to_list(columns=["start", "stop"])
    start_stop_nonoverlapping = cogent3.util.misc.get_merged_overlapping_coords(start_stop_overlapping)
    #using subsample to speed analysis, remove next line when doing the final analysis
    
    inter_genic = []
    last_end = 0
    for start, stop in start_stop_nonoverlapping:
       inter_genic.append((ref_species, chrom_coord, last_end, start-1, strand))
       last_end = stop
    
    intergen_tab = cogent3.make_table(header=["species", "seqid", "start", "stop", "strand"], data=inter_genic)
    intergen_tab.write(folder_out + "chrom" + str(chrom_coord) + "_intergenic.tsv")

    #Splitting table into proximal and distal sites
    proximal5_IG = []
    proximal3_IG = []
    distal_IG = []
    
    pd_intergenic_tab = intergen_tab.to_pandas()
    intergeniclengths = pd_intergenic_tab["stop"] - pd_intergenic_tab["start"]
    for i in range(len(pd_intergenic_tab)):
        start_pos = pd_intergenic_tab["start"][i]
        end_pos = pd_intergenic_tab["stop"][i]
    
        #at the begining of a chromosome there is no proximal 3'
        if start_pos == 0:
            if intergeniclengths[i] < proximal_distance/2:
                proximal5_IG.append((ref_species, chrom_coord, start_pos, end_pos, strand))
            else:
                distal_IG.append((ref_species, chrom_coord, start_pos, end_pos - proximal_distance - 1, strand))
                proximal5_IG.append((ref_species, chrom_coord, end_pos - proximal_distance, end_pos, strand))
    
        #intergenic region is too short
        #divide the region in half. First half counts as proximal3_IG and the second as proximal3_IG
        else:
            if intergeniclengths[i] < proximal_distance:
                middle_point_fromstart = intergeniclengths[i]/2
                proximal3_IG.append((ref_species, chrom_coord, start_pos, round(start_pos + middle_point_fromstart), strand))
                proximal5_IG.append((ref_species, chrom_coord, round(start_pos + middle_point_fromstart), end_pos, strand))
        
            else:
                proximal3_IG.append((ref_species, chrom_coord, start_pos, start_pos + proximal_distance, strand))
                distal_IG.append((ref_species, chrom_coord, start_pos + proximal_distance, end_pos - proximal_distance - 1, strand))
                proximal5_IG.append((ref_species, chrom_coord, end_pos - proximal_distance, end_pos, strand))
    
    proximal3_IG = cogent3.make_table(header=["species", "seqid", "start", "stop", "strand"], data=proximal3_IG)
    proximal3_IG.write(folder_out + "chrom" + str(chrom_coord) + "_proximal3_IG_coordinates.tsv")
    
    distal_IG = cogent3.make_table(header=["species", "seqid", "start", "stop", "strand"], data=distal_IG)
    distal_IG.write(folder_out + "chrom" + str(chrom_coord) + "_distal_IG_coordinates.tsv")
    
    proximal5_IG = cogent3.make_table(header=["species", "seqid", "start", "stop", "strand"], data=proximal5_IG)
    proximal5_IG.write(folder_out + "chrom" + str(chrom_coord) + "_proximal5_IG_coordinates.tsv")    